In [ ]:
"""
This Python notebook reproduces panels Figure 4D, 4E, 4G, and 4H from
the IHEC “Flagship” paper, as detailed below.

4D:
Boxplots showing the accuracy and macro F1-score of 10-fold cross-validated MLP classifiers
trained to predict biospecimen (cell type) metadata using multiple versions of the input
features. Metrics are also reported separately for each experimental assay. Boxes represent
the interquartile range (Q1–Q3), with the median shown as a solid line and the mean as a
dashed line. Whiskers extend to the most extreme data points within 1.5 times the interquartile
range.

For Figures 4E, 4G, and 4H, classifiers were trained using the full feature set (whole
genome, 100 kb bins). Important features for each biospecimen class were identified using
SHAP values.

4E:
Table summarizing GO-term enrichments associated with the important classifier features for
each biospecimen. Enrichments were computed with g:Profiler.

4G:
Violin plots showing the distribution of the average maximum ChromScore for important
100 kb bins across biospecimens. For each biospecimen, the relevant files are selected, and
for each important bin, the mean of its maximum ChromScore across those files is computed.
Each violin contains as many points as there are important bins for that biospecimen. Black
boxes indicate Q1–Q3.

4H:
Stacked bar plots showing the proportion of high- and low-confidence predictions (scores
≥0.6 vs <0.6) for samples with missing metadata labels, drawn from recount3 and ChIP-Atlas.

For full methodological details on how each value was computed, see the complete notebook,
which includes steps starting from much closer to the raw results.
https://github.com/labjacquespe/EpiClass/blob/master/src/python/epiclass/utils/notebooks/paper/paper-final/flagship_figures.ipynb
"""

# pylint: disable=import-error, redefined-outer-name, use-dict-literal, too-many-lines, too-many-branches, duplicate-code
from __future__ import annotations

## SETUP

In [ ]:
import copy
import json
from pathlib import Path
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import display
from plotly.subplots import make_subplots
from scipy import stats

In [ ]:
pio.renderers.default = "notebook"

In [ ]:
ASSAY = "assay_epiclass"
CELL_TYPE = "harmonized_sample_ontology_intermediate"

In [ ]:
base_fig_dir = Path.home() / "projects" / "epiclass" / "output" / "paper" / "figures"
plot_data_dir = base_fig_dir / "flagship" / "pre-plot"
if not plot_data_dir.exists():
    raise FileNotFoundError(f"Plot data directory not found: {plot_data_dir}")

fig_N = "4"

In [ ]:
class IHECColorMap:
    """Class to handle IHEC color map."""

    def __init__(self, color_map_path: Path):
        with open(color_map_path, "r", encoding="utf8") as f:
            self.ihec_color_map = json.load(f)
        self.assay_color_map = self.create_assay_color_map()
        self.cell_type_color_map = self.create_cell_type_color_map()

    def _create_color_map(self, label_category: str) -> Dict[str, str]:
        """Create a rbg color map from IHEC rgb strings"""
        color_dict = [elem for elem in self.ihec_color_map if label_category in elem][0][
            label_category
        ][0]
        for name, color in list(color_dict.items()):
            rbg = color.split(",")
            color_dict[name] = f"rgb({rbg[0]},{rbg[1]},{rbg[2]})"
            color_dict[name.lower()] = color_dict[name]
            color_dict[name.lower().replace("-", "_")] = color_dict[name]
        return color_dict

    def create_assay_color_map(self) -> Dict[str, str]:
        """Create a rbg color map for ihec assays."""
        category_label = "experiment"
        color_dict = self._create_color_map(category_label)
        color_dict["mrna_seq"] = color_dict["rna_seq"]
        for assay in ["wgbs-pbat", "wgbs-standard"]:
            color_dict[assay] = color_dict["wgbs"]
            color_dict[assay.replace("-", "_")] = color_dict["wgbs"]
        return color_dict

    def create_cell_type_color_map(self) -> Dict[str, str]:
        """Create the rbg color map for ihec cell types."""
        return self._create_color_map(CELL_TYPE)

In [ ]:
ihec_color_map_path = plot_data_dir / "IHEC_EpiATLAS_IA_colors_Apl01_2024.json"
IHECColorMap = IHECColorMap(ihec_color_map_path)
assay_colors = IHECColorMap.assay_color_map
cell_type_colors = IHECColorMap.cell_type_color_map

## Panel D - Cell type accuracy per assay, for different training regions

In [ ]:
resolution_colors = {
    "100kb": px.colors.qualitative.Safe[0],
    "10kb": px.colors.qualitative.Safe[1],
    "1kb": px.colors.qualitative.Safe[2],
    "regulReg": px.colors.qualitative.Safe[3],
    "gene": px.colors.qualitative.Safe[4],
    "cpg": px.colors.qualitative.Safe[5],
    "1mb": px.colors.qualitative.Safe[6],
    "5mb": px.colors.qualitative.Safe[7],
    "10mb": px.colors.qualitative.Safe[8],
}

#### Global Average

In [ ]:
def graph_feature_set_metrics(
    all_metrics: Dict[str, Dict[str, Dict[str, Dict[str, float]]]],
    input_sizes: Dict[str, int],
    logdir: Path | str | None = None,
    sort_by_input_size: bool = False,
    name: str | None = None,
    y_range: Tuple[float, float] | None = None,
    boxpoints: str = "all",
    width: int = 1200,
    height: int = 1200,
    verbose: bool = False,
) -> None:
    """Graph the metrics for all feature sets.

    Args:
        all_metrics (Dict[str, Dict[str, Dict[str, Dict[str, float]]]): A dictionary containing all metrics for all feature sets.
            Format: {feature_set: {task_name: {split_name: metric_dict}}}
        input_sizes (Dict[str, int]): A dictionary containing the input sizes for all feature sets.
        logdir (Path): The directory where the figure will be saved. If None, the figure will only be displayed.
        sort_by_input_size (bool): Whether to sort the feature sets by input size.
        name (str|None): The name of the figure.
        y_range (Tuple[float, float]|None): The y-axis range for the figure.
        boxpoints (str): The type of boxpoints to display. Can be "all" or "outliers". Defaults to "all".
    """
    if boxpoints not in ["all", "outliers"]:
        raise ValueError("Invalid boxpoints value.")

    reference_hdf5_type = "hg38_100kb_all_none"
    metadata_categories = list(all_metrics[reference_hdf5_type].keys())

    used_resolutions = set()
    for i in range(len(metadata_categories)):
        category_idx = i
        category_fig = make_subplots(
            rows=1,
            cols=2,
            shared_yaxes=True,
            subplot_titles=["Accuracy", "F1-score (macro)"],
            # x_title="Feature set",
            y_title="Metric value",
        )

        trace_names = []
        order = list(all_metrics.keys())
        if sort_by_input_size:
            order = sorted(
                all_metrics.keys(),
                key=lambda x: input_sizes[x],
            )
        for feature_set_name in order:
            if verbose:
                print(f"Processing feature set: {feature_set_name}")

            tasks_dicts = all_metrics[feature_set_name]
            meta_categories = copy.deepcopy(metadata_categories)

            if feature_set_name not in input_sizes:
                print(f"Skipping {feature_set_name}, no input size found.")
                continue

            task_name = meta_categories[category_idx]
            if "split" in task_name:
                raise ValueError("Split in task name. Wrong metrics dict.")

            try:
                task_dict = tasks_dicts[task_name]
            except KeyError as err:
                if verbose:
                    print(f"KeyError for {feature_set_name}, {task_name}: {err}")
                else:
                    print("Skipping", feature_set_name, task_name)
                    continue

            input_size = input_sizes[feature_set_name]

            feature_set_name = feature_set_name.replace("_none", "")
            feature_set_name = feature_set_name.replace("hg38_", "")

            resolution = feature_set_name.split("_")[0]
            used_resolutions.add(resolution)

            trace_name = f"{input_size}|{feature_set_name}"
            trace_names.append(trace_name)

            # Accuracy
            metric = "Accuracy"
            y_vals = [task_dict[split][metric] for split in task_dict]
            hovertext = [
                f"{split}: {metrics_dict[metric]:.4f}"
                for split, metrics_dict in task_dict.items()
            ]
            category_fig.add_trace(
                go.Box(
                    y=y_vals,
                    name=trace_name,
                    boxmean=True,
                    boxpoints=boxpoints,
                    marker=dict(size=3, color="black"),
                    line=dict(width=1, color="black"),
                    fillcolor=resolution_colors[resolution],
                    hovertemplate="%{text}",
                    text=hovertext,
                    legendgroup=resolution,
                    showlegend=False,
                ),
                row=1,
                col=1,
            )

            metric = "F1_macro"
            y_vals = [task_dict[split][metric] for split in task_dict]
            hovertext = [
                f"{split}: {metrics_dict[metric]:.4f}"
                for split, metrics_dict in task_dict.items()
            ]
            category_fig.add_trace(
                go.Box(
                    y=y_vals,
                    name=trace_name,
                    boxmean=True,
                    boxpoints=boxpoints,
                    marker=dict(size=3, color="black"),
                    line=dict(width=1, color="black"),
                    fillcolor=resolution_colors[resolution],
                    hovertemplate="%{text}",
                    text=hovertext,
                    legendgroup=resolution,
                    showlegend=False,
                ),
                row=1,
                col=2,
            )

        title = f"Neural network performance - {metadata_categories[category_idx]}"
        if name is not None:
            title += f" - {name}"
        category_fig.update_layout(
            width=width,
            height=height,
            title=title,
        )

        # dummy scatters for resolution colors
        for resolution, color in resolution_colors.items():
            if resolution not in used_resolutions:
                continue
            category_fig.add_trace(
                go.Scatter(
                    x=[None],
                    y=[None],
                    mode="markers",
                    name=resolution,
                    marker=dict(color=color, size=5),
                    showlegend=True,
                    legendgroup=resolution,
                )
            )

        category_fig.update_layout(legend=dict(itemsizing="constant"))

        # y-axis
        if y_range:
            category_fig.update_yaxes(range=y_range)
        else:
            if ASSAY in task_name:
                category_fig.update_yaxes(range=[0.96, 1.001])
            if CELL_TYPE in task_name:
                category_fig.update_yaxes(range=[0.75, 1])

        # Save figure
        if logdir:
            logdir = Path(logdir)
            base_name = f"feature_set_metrics_{metadata_categories[category_idx]}"
            if name is not None:
                base_name = base_name + f"_{name}"
            category_fig.write_html(logdir / f"{base_name}.html")
            category_fig.write_image(logdir / f"{base_name}.svg")
            category_fig.write_image(logdir / f"{base_name}.png")

        category_fig.show()

In [ ]:
all_metrics_path = plot_data_dir / f"Fig{fig_N}_D_metrics.json"
with open(all_metrics_path, "r", encoding="utf8") as f:
    all_metrics = json.load(f)

input_sizes_path = plot_data_dir / f"Fig{fig_N}_D_input_sizes.json"
with open(input_sizes_path, "r", encoding="utf8") as f:
    input_sizes = json.load(f)

In [ ]:
graph_feature_set_metrics(
    all_metrics=all_metrics,  # type: ignore
    input_sizes=input_sizes,
    boxpoints="all",
    width=700,
    height=800,
    y_range=(0.295, 1.005),
)

#### Metrics per assay

In [ ]:
def graph_feature_set_metrics_per_assay(
    all_metrics_per_assay: Dict[str, Dict[str, Dict[str, Dict[str, Dict[str, float]]]]],
    input_sizes: Dict[str, int],
    logdir: Path | None = None,
    sort_by_input_size: bool = False,
    name: str | None = None,
    y_range: Tuple[float, float] | None = None,
    boxpoints: str = "outliers",
) -> None:
    """Graph the metrics for all feature sets, per assay, with separate plots for accuracy and F1-score.

    Args:
        all_metrics_per_assay (Dict[str, Dict[str, Dict[str, Dict[str, Dict[str, float]]]]]): A dictionary containing all metrics per assay for all feature sets.
            Format: {assay: {feature_set: {task_name: {split_name: metric_dict}}}}
        input_sizes (Dict[str, int]): A dictionary containing the input sizes for all feature sets.
        logdir (Path): The directory where the figures will be saved. If None, the figures will only be displayed.
        sort_by_input_size (bool): Whether to sort the feature sets by input size.
        name (str|None): The name of the figure.
        y_range (Tuple[float, float]|None): The y-axis range for the plots.
        boxpoints (str): The type of points to display in the box plots. Defaults to "outliers".
    """
    valid_boxpoints = ["all", "outliers"]
    if boxpoints not in valid_boxpoints:
        raise ValueError(f"Invalid boxpoints value. Choose from {valid_boxpoints}.")

    fig_assay_order = [
        "rna_seq",
        "h3k27ac",
        "h3k4me1",
        "h3k4me3",
        "h3k36me3",
        "h3k27me3",
        "h3k9me3",
        "input",
        "wgbs",
    ]

    reference_assay = next(iter(all_metrics_per_assay))
    reference_feature_set = next(iter(all_metrics_per_assay[reference_assay]))
    metadata_categories = list(
        all_metrics_per_assay[reference_assay][reference_feature_set].keys()
    )

    for _, category in enumerate(metadata_categories):
        for metric, metric_name in [
            ("Accuracy", "Accuracy"),
            ("F1_macro", "F1-score (macro)"),
        ]:
            fig = go.Figure()

            feature_sets = list(all_metrics_per_assay[reference_assay].keys())
            unique_feature_sets = set(feature_sets)
            for assay in fig_assay_order:
                if set(all_metrics_per_assay[assay].keys()) != unique_feature_sets:
                    raise ValueError("Different feature sets through assays.")

            feature_set_order = feature_sets
            if sort_by_input_size:
                feature_set_order = sorted(
                    feature_set_order, key=lambda x: input_sizes[x]
                )

            # Adjust spacing so each assay group has dedicated space based on the number of feature sets
            spacing_multiplier = (
                1.1  # Increase this multiplier if needed to add more spacing
            )
            x_positions = {
                assay: i * len(feature_set_order) * spacing_multiplier
                for i, assay in enumerate(fig_assay_order)
            }

            for i, feature_set_name in enumerate(feature_set_order):
                resolution = (
                    feature_set_name.replace("_none", "")
                    .replace("hg38_", "")
                    .split("_")[0]
                )
                color = resolution_colors[resolution]
                display_name = feature_set_name.replace("_none", "").replace("hg38_", "")

                for assay in fig_assay_order:
                    if feature_set_name not in all_metrics_per_assay[assay]:
                        continue

                    tasks_dicts = all_metrics_per_assay[assay][feature_set_name]

                    if feature_set_name not in input_sizes:
                        print(f"Skipping {feature_set_name}, no input size found.")
                        continue

                    task_name = category
                    if "split" in task_name:
                        raise ValueError("Split in task name. Wrong metrics dict.")

                    try:
                        task_dict = tasks_dicts[task_name]
                    except KeyError:
                        print(
                            f"Skipping {feature_set_name}, {task_name} for assay {assay}"
                        )
                        continue

                    y_vals = [task_dict[split][metric] for split in task_dict]
                    hovertext = [
                        f"{assay} - {display_name} - {split}: {metrics_dict[metric]:.4f}"
                        for split, metrics_dict in task_dict.items()
                    ]

                    x_position = x_positions[assay] + i
                    fig.add_trace(
                        go.Box(
                            x=[x_position] * len(y_vals),
                            y=y_vals,
                            name=f"{assay}|{display_name}",
                            boxmean=True,
                            boxpoints=boxpoints,
                            marker=dict(size=3, color="black"),
                            line=dict(width=1, color="black"),
                            fillcolor=color,
                            hovertemplate="%{text}",
                            text=hovertext,
                            showlegend=False,
                            legendgroup=display_name,
                        )
                    )

                    # separate box groups
                    fig.add_vline(
                        x=x_positions[assay] - 1, line_width=1, line_color="black"
                    )

            # Add dummy traces for the legend
            for feature_set_name in feature_set_order:
                resolution = (
                    feature_set_name.replace("_none", "")
                    .replace("hg38_", "")
                    .split("_")[0]
                )
                color = resolution_colors[resolution]
                display_name = feature_set_name.replace("_none", "").replace("hg38_", "")

                fig.add_trace(
                    go.Scatter(
                        name=display_name,
                        x=[None],
                        y=[None],
                        mode="markers",
                        marker=dict(size=10, color=color),
                        showlegend=True,
                        legendgroup=display_name,
                    )
                )

            title = f"Neural network performance - {category} - {metric_name} (per assay)"
            if name is not None:
                title += f" - {name}"
            fig.update_layout(
                width=1500,
                height=1000,
                title=title,
                xaxis_title="Assay",
                yaxis_title=metric_name,
            )

            # Create x-axis labels
            fig.update_xaxes(
                tickmode="array",
                tickvals=[
                    x_positions[assay] + len(feature_set_order) / 2
                    for assay in fig_assay_order
                ],
                ticktext=list(x_positions.keys()),
                title="Assay",
            )

            fig.update_layout(
                legend=dict(
                    title="Feature Sets", itemsizing="constant", traceorder="normal"
                )
            )
            if y_range:
                fig.update_yaxes(range=y_range)

            if logdir:
                base_name = f"feature_set_metrics_{category}_{metric}_per_assay"
                if name is not None:
                    base_name = base_name + f"_{name}"
                fig.write_html(logdir / f"{base_name}.html")
                fig.write_image(logdir / f"{base_name}.svg")
                fig.write_image(logdir / f"{base_name}.png")

            fig.show()

In [ ]:
all_metrics_path = plot_data_dir / f"Fig{fig_N}_D_metrics_per_assay.json"
with open(all_metrics_path, "r", encoding="utf8") as f:
    metrics_per_assay = json.load(f)

In [ ]:
graph_feature_set_metrics_per_assay(
    all_metrics_per_assay=metrics_per_assay,  # type: ignore
    input_sizes=input_sizes,
    boxpoints="all",
    sort_by_input_size=False,
    y_range=(0.295, 1.005),
)

## Panel E - SHAP values: Gene ontology enrichment analysis

### Prep data

In [ ]:
selected_cell_types = [
    "T_cell",
    "neutrophil",
    "lymphocyte_of_B_lineage",
    "brain",
]
go_terms_table = [
    "T cell receptor complex",
    "plasma membrane signaling receptor complex",
    "adaptive immune response",
    # "receptor complex",
    "secretory granule",
    "secretory vesicle",
    "secretory granule membrane",
    # "intracellular vesicle",
    "immunoglobulin complex",
    "immune response",
    # "immune system process",
    "homophilic cell adhesion via plasma membrane adhesion molecules",
    "DNA binding",
    "cell-cell adhesion via plasma-membrane adhesion molecules",
    # "RNA polymerase II cis-regulatory region sequence-specific DNA binding",
    # "blood microparticle",
    # "platelet alpha granule lumen",
    # "fibrinogen complex",
    # "endoplasmic reticulum lumen",
]

In [ ]:
preplot_path = plot_data_dir / f"Fig{fig_N}_E_concat_gprofiler.tsv"
full_concat_df = pd.read_csv(preplot_path, sep="\t")

In [ ]:
table = full_concat_df.pivot_table(
    index="name", columns="shap_source", values="table_val", aggfunc="mean"
)

### Graph

In [ ]:
# include line break for long GO terms
go_terms_graph = [
    "T cell receptor complex",
    "plasma membrane<br>signaling receptor complex",
    "adaptive immune response",
    # "receptor complex",
    "secretory granule",
    "secretory vesicle",
    "secretory granule membrane",
    # "intracellular vesicle",
    "immunoglobulin complex",
    "immune response",
    # "immune system process",
    "homophilic cell adhesion via<br>plasma membrane adhesion molecules",
    "DNA binding",
    "cell-cell adhesion via<br>plasma-membrane adhesion molecules",
    #     "RNA polymerase II cis-regulatory<br>region sequence-specific DNA binding",
    #     "blood microparticle",
    #     "platelet alpha granule lumen",
    #     "fibrinogen complex",
    #     "endoplasmic reticulum lumen",
]

In [ ]:
# Keep only selected cell types and GO terms
sub_table = table.loc[go_terms_table, selected_cell_types].copy()
assert sub_table.shape == (len(go_terms_graph), len(selected_cell_types))

In [ ]:
# Rename the GO terms for better visualization
sub_table = sub_table.rename(index=dict(zip(go_terms_table, go_terms_graph)))

In [ ]:
sigma = "\u03c3"

colorbar = dict(
    title="-log<sub>10</sub>(p-value)",
    tickvals=[0, 1.30, 2, 3, 5, 6.53, 10],
    ticktext=[
        "0",
        f"1.30: p=0.05~2{sigma}",
        "2: p=0.01",
        f"3: p=0.001~3{sigma}",
        "5",
        f"6.53: p=3x10<sup>-7</sup> = 5{sigma}",
        "10",
    ],
)

In [ ]:
# Fill z with NaNs (for color) and make custom text labels
z = sub_table.values  # keep NaNs in for color
text = np.where(
    np.isnan(z), "NS", np.char.mod("%.2f", z)  # format floats to 2 decimal places
)

fig = go.Figure(
    data=go.Heatmap(
        z=np.nan_to_num(
            z, nan=0
        ),  # replace NaNs with 0 for coloring (or use a custom colormap later)
        x=sub_table.columns,
        y=sub_table.index,
        colorscale="Blues",
        zmin=0,
        zmax=10,
        colorbar=colorbar,
        text=text,
        texttemplate="%{text}",
        hovertemplate="GO Term: %{y}<br>Class: %{x}<br>Value: %{z:.2f}<extra></extra>",
        showscale=True,
        xgap=2,
        ygap=2,
    )
)

# Customize layout
fig.update_layout(
    width=600,
    height=800,
    plot_bgcolor="black",
    margin=dict(t=120),
    title={
        "text": "Top SHAP regions: GO term enrichment",
        "y": 0.99,  # Position closer to top
        "x": 0.01,
        "xanchor": "left",
        "yanchor": "top",
    },
)
# Fix gridlines
fig.update_xaxes(showgrid=False, side="top")
fig.update_yaxes(showgrid=False, autorange="reversed")

fig.update_layout()

fig.show()

## Panel G - Max Chromscore for important SHAP features for certain cell types

Global max chromscore vs per cell type, for important SHAP regions only

Same cell types as in panel E + hepatocyte

In [ ]:
def test_distribution(
    x: List[float], y: List[float], verbose: bool = True
) -> Tuple[float, float]:
    """Test for distribution difference. x is used as reference for the number of samples.

    Welch's t-test and Brunner-Munzel test are computed.

    Returns:
        Tuple[float, float]: p-value for each test
    """
    if verbose:
        print(f"Number of samples in x: {len(x)}")
        print(f"Number of samples in y: {len(y)}")

    Welch_pval = stats.ttest_ind(
        a=x,
        b=y,
        equal_var=False,
        alternative="two-sided",
        nan_policy="raise",
    ).pvalue  # type: ignore

    BM_pval = stats.brunnermunzel(
        x,
        y,
        alternative="two-sided",
        nan_policy="raise",
        distribution="t",
    ).pvalue

    return Welch_pval, BM_pval


def define_pval_label(pval: float) -> str:
    """Define p-value label."""
    pval_symbol = ""
    if pval < 0.001:
        pval_symbol = "<0.001***"
    elif pval < 0.01:
        pval_symbol = "<0.01**"
    elif pval < 0.05:
        pval_symbol = "<0.05*"
    elif pval >= 0.05:
        pval_symbol = ">0.05 NS"

    return pval_symbol

In [ ]:
def plot_chromscore_per_biospecimen_violin(
    graph_data: Dict[str, Dict[str, List[float] | int]],
    cell_types: List[str] | None = None,
    logdir: Path | None = None,
    do_subplots: bool = True,
    filename: str = "important_features_16ct_max_chromscore_100kb_per_biospecimen_2violin",
    verbose: bool = False,
) -> pd.DataFrame:
    """
    Plot average of 'max chromscore' per biospecimen as violin plots,
    using regions and files per biospecimen independently.

    The average is computed over files, so one averaged value per feature/bin/region.

    Args:
        graph_data: Dict[str, Dict[str, List[float] | int]]. From prepare_chromscore_per_biospecimen_data.
        cell_types: List[str]|None. List of cell types to plot.

    Returns:
        pd.DataFrame. Dataframe with pvals.
    """
    data = copy.deepcopy(graph_data)
    if not cell_types:
        cell_types = list(data.keys())
        cell_types.remove("all_files")

    colors = px.colors.qualitative.Dark24[0:2]

    fig = go.Figure()
    if do_subplots:
        fig = make_subplots(
            rows=4,
            cols=4,
            shared_yaxes=True,
            vertical_spacing=0.075,
            horizontal_spacing=0.025,
            y_title="Average of max value in selected regions of 100kb (over files)",
        )

    # Filter
    try:
        data = {biospecimen: graph_data[biospecimen] for biospecimen in cell_types}
    except KeyError as err:
        raise KeyError(
            f"A cell type is missing from the graph_data.\ncell types: {graph_data.keys()}.\nDesired: {cell_types}."
        ) from err

    all_pvals = []
    trace_names = []
    for idx, (biospecimen, data) in enumerate(data.items()):
        if biospecimen not in cell_types:
            continue

        avg_per_bin: List[float] = data["avg_per_bin"]  # type: ignore
        all_means_file_subset: List[float] = data["all_means_file_subset"]  # type: ignore

        nb_files = data["nb_files"]
        nb_features = len(avg_per_bin)

        if do_subplots:
            placement_dict = {
                "row": idx // 4 + 1,
                "col": idx % 4 + 1,
            }
        else:
            placement_dict = {}

        # Important features
        show_points = False
        if len(avg_per_bin) <= 10:
            show_points = "all"
        fig.add_trace(
            go.Violin(
                side="negative",
                name=f"trace{idx}",
                y=avg_per_bin,
                fillcolor=colors[0],
                line=dict(color="black", width=1.5 if do_subplots else 0),
                showlegend=False,
                meanline_visible=True,
                points=show_points,
                spanmode="hard",
                legendgroup="All features",
                box=dict(
                    visible=True,
                    fillcolor=colors[0] if do_subplots else "black",
                    width=0.4,
                    line_width=0.5 if do_subplots else 0,
                ),
                scalemode="width",  # occupy all possible space for subplots
                scalegroup=f"trace{idx}",
            ),
            **placement_dict,  # type: ignore
        )

        # Global distribution comparison
        fig.add_trace(
            go.Violin(
                side="positive",
                name=f"trace{idx}",
                y=all_means_file_subset,
                fillcolor=colors[1],
                line=dict(color="black", width=1.5 if do_subplots else 0),
                showlegend=False,
                meanline_visible=True,
                points=False,
                spanmode="hard",
                legendgroup="All features",
                box=dict(
                    visible=True,
                    fillcolor=colors[1] if do_subplots else "black",
                    width=0.4,
                    line_width=0.5 if do_subplots else 0,
                ),
                scalemode="width",
                scalegroup=f"trace{idx}",
            ),
            **placement_dict,  # type: ignore
        )

        pvals = test_distribution(
            x=avg_per_bin,
            y=all_means_file_subset,
            verbose=False,
        )
        if verbose:
            print(f"{biospecimen}, {nb_features} features, {nb_files} files")
            print(f"pvals [Welch, BM]: {pvals}\n\n")

        all_pvals.append(
            [biospecimen, nb_files, nb_features, len(all_means_file_subset), *pvals]
        )

        pval = float(np.max(pvals))
        pval_symbol = define_pval_label(pval)

        if do_subplots:
            group_name = f"{biospecimen}<br>({nb_files} files, {nb_features} features)<br>p{pval_symbol}"
            fig.update_xaxes(
                showticklabels=False,
                row=idx // 4 + 1,
                col=idx % 4 + 1,
                title=group_name,
                title_standoff=2,
                title_font=dict(size=10),
            )
        else:
            group_name = f"{biospecimen} ({nb_files} files, {nb_features} features), p{pval_symbol}"
            trace_names.append(group_name)

    # Manually set names for traces
    if not do_subplots:
        newnames = {f"trace{idx}": name for idx, name in enumerate(trace_names)}
        fig.for_each_trace(lambda t: t.update(name=newnames[t.name]))

    # Legend with dummy points
    for i, name in enumerate(["Important SHAP features", "All features"]):
        fig.add_trace(
            go.Scatter(
                x=[None],
                y=[None],
                mode="markers",
                name=name,
                legendgroup=name,
                showlegend=True,
                marker=dict(color=colors[i], symbol="square"),
            ),
        )

    fig.update_yaxes(range=[0, 1])

    fig.update_layout(
        title="ChromScore per biospecimen file subset",
        width=1000,
        height=900,
        legend=dict(
            itemsizing="constant",
            yanchor="top",
            xanchor="right",
            y=1.1,
            x=0.8,
        ),
    )

    # fig.update_layout(violingap=0, violinmode='overlay')

    if not do_subplots:
        fig.update_layout(
            yaxis_title="Average of max value in selected regions of 100kb (over files)",
            xaxis_title="Biospecimen",
            width=700,
            height=700,
        )

    fig.show()

    if logdir is not None:
        print("Saving figure.")
        fig.write_image(logdir / f"{filename}.svg")
        fig.write_image(logdir / f"{filename}.png", scale=1.5)
        fig.write_html(logdir / f"{filename}.html")

    return pd.DataFrame(
        all_pvals,
        columns=[
            "biospecimen",
            "nb_files",
            "Nb features (N_1)",
            "Nb features global (N_2)",
            "pval_Welch",
            "pval_BM",
        ],
    )

In [ ]:
cell_types = ["t_cell", "neutrophil", "lymphocyte_of_b_lineage", "brain", "hepatocyte"]

In [ ]:
preplot_path = (
    plot_data_dir / f"Fig{fig_N}_G_max_chromscore_important_bins_per_cell_type.json"
)
with open(preplot_path, "r", encoding="utf-8") as f:
    graph_data = json.load(f)

In [ ]:
pvals_df = plot_chromscore_per_biospecimen_violin(
    graph_data=graph_data,
    do_subplots=False,
    cell_types=cell_types,
)

Notes: 
- The above p-values of the graph are not corrected for multiple hypothesis testing. Using a Bonferroni correction for 5 tests, all remain significant (at p < 0.05). See table below for corrected p-values. If we apply for 16 tests (all classifier cell types), then the hepatocyte p-value becomes closer to 0.05
- The max chromscore values were computed directly from bigwig files using pyBigWig, for each 100kb bin, and then, for each bin/region, those max values were averaged across the relevant cell type files (e.g. for T cells, across all T cell files). The number of point per violin corresponds to the number of important SHAP regions/features for that cell type. This is compared to the average max chromscore across the same files, for all 30321 100kb bins. This means the left violing is a subset of the right violin.

In [ ]:
df = pvals_df.set_index("biospecimen")

df["corrected_pval_Welch"] = df["pval_Welch"].apply(lambda x: min(1, x * len(df)))
df["corrected_pval_BM"] = df["pval_BM"].apply(lambda x: min(1, x * len(df)))

display(df)

## Panel H - Inference on other public sources

Namely, ChIP-Atlas and recount3.

In [ ]:
preplot_path = plot_data_dir / f"Fig{fig_N}_H_metrics_per_assay_public_DB.tsv"
summary_df = pd.read_csv(preplot_path, sep="\t")

In [ ]:
# we want extract low/high confidence pred per DB (>=0.6), and missing labels
assay_epiclass_labels = ["avg-all", "count-unknown"]
summary_df = summary_df[summary_df["assay_epiclass"].isin(assay_epiclass_labels)]

In [ ]:
categories_to_plot = ["cancer_status", "sex", "biomaterial_type"]

# Formatting data for plotting
graph_values = []
for category in categories_to_plot:
    df = summary_df[summary_df["task_name"] == category]

    # Known labels
    # nb known labels is going to be number of samples for avg-all min_predScore => 0.0
    N_known_labels = df[
        (df["assay_epiclass"] == "avg-all") & (df["min_predScore"].astype(str) == "0.0")
    ]["nb_samples"].sum()
    graph_values.append([category, N_known_labels, "Provided/Extracted"])

    # Unknown labels
    unknown_df_cond = df["assay_epiclass"] == "count-unknown"
    all_pred_cond = df["min_predScore"].astype(str) == "0.0"
    high_pred_cond = df["min_predScore"].astype(str) == "0.6"

    N_total_unknown = df[unknown_df_cond & all_pred_cond]["nb_samples"].sum()
    N_high_conf = df[unknown_df_cond & high_pred_cond]["nb_samples"].sum()

    N_low_conf = N_total_unknown - N_high_conf

    graph_values.append([category, N_high_conf, "High Conf Pred"])
    graph_values.append([category, N_low_conf, "Low Conf Pred"])

plot_df = pd.DataFrame(data=graph_values, columns=["Category", "Count", "Type"])

In [ ]:
# Define colors for graphs
color_map = {
    "Low Conf Pred": "#7fccd6",  # light blue
    "High Conf Pred": "#004e58",  # dark blue
    "Provided/Extracted": "#000000",  # black
}

In [ ]:
# Plot percentage of samples per category and type
fig_pct = px.bar(
    plot_df,
    x="Category",
    y="Count",
    color="Type",
    color_discrete_map=color_map,
    title="Data Availability and Prediction Confidence",
    labels={"Count": "Percentage of public data samples (%)"},
    category_orders={"Type": ["Provided/Extracted", "High Conf Pred", "Low Conf Pred"]},
)

# Update layout to normalize bars to 100%
fig_pct.update_layout(
    barnorm="percent",
    height=600,
    width=700,
    legend_traceorder="reversed",
)

fig_pct.update_yaxes(ticksuffix="%", range=[0, 101])

fig_pct.show()

In [ ]:
# Plot number of samples per category and type
fig = px.bar(
    plot_df,
    x="Category",
    y="Count",
    color="Type",
    color_discrete_map=color_map,
    title="Data Availability and Prediction Confidence",
    labels={"Count": "Number of Samples"},
    category_orders={"Type": ["Provided/Extracted", "High Conf Pred", "Low Conf Pred"]},
)

fig.update_layout(
    barnorm=None,
    height=600,
    width=700,
    legend_traceorder="reversed",
)

fig.show()